In [3]:
"""
MIVA PHS 410 Capstone: Biostatistical Analysis Pipeline
Study: Socioeconomic, Healthcare, and Environmental Determinants of Childhood Stunting in Nigeria
Dataset: MICS6 Simulation Cohort (Children Under 5)

"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ==============================================================================
# 1. LOAD DATASET & DATA WRANGLING
# ==============================================================================
# Load the raw dataset
df = pd.read_csv("MIVA_PHS410_DS1_MICS_Extract_SIMULATED.csv")

# Exclude records with missing anthropometry/stunting (n = 110)
df_clean = df.dropna(subset=["STUNTED"]).copy()
df_clean["STUNTED"] = df_clean["STUNTED"].astype(int)

# Map area of residence (HH6: 1=Urban, 2=Rural)
df_clean["residence"] = df_clean["HH6"].map({1: "Urban", 2: "Rural"})

# Map geopolitical zone (HH7: 1-6)
zone_map = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West",
}
df_clean["zone"] = df_clean["HH7"].map(zone_map)

# Map child sex (HL4: 1=Male, 2=Female)
df_clean["sex"] = df_clean["HL4"].map({1: "Male", 2: "Female"})

# Discretize child age into standard epidemiological cohorts (CAGE in months)
age_bins = [-1, 11, 23, 35, 47, 59]
age_labels = [
    "0–11 months",
    "12–23 months",
    "24–35 months",
    "36–47 months",
    "48–59 months",
]
df_clean["age_group"] = pd.cut(
    df_clean["CAGE"], bins=age_bins, labels=age_labels
)

# Map maternal education level (melevel: 0=None, 1=Primary, 2=Secondary, 3=Higher)
edu_map = {0: "None", 1: "Primary", 2: "Secondary", 3: "Higher"}
df_clean["maternal_edu"] = df_clean["melevel"].map(edu_map)

# Discretize maternal age (mage in years)
mage_bins = [14, 24, 34, 50]
mage_labels = ["15–24 years", "25–34 years", "35–49 years"]
df_clean["maternal_age_grp"] = pd.cut(
    df_clean["mage"], bins=mage_bins, labels=mage_labels
)

# Map wealth quintiles (windex5: 1-5)
wealth_map = {
    1: "Poorest",
    2: "Second",
    3: "Middle",
    4: "Fourth",
    5: "Richest",
}
df_clean["wealth_quintile"] = df_clean["windex5"].map(wealth_map)

# Map antenatal care visits based on WHO benchmark (<4 vs >=4 visits)
df_clean["anc_cat"] = np.where(
    df_clean["ANCVISITS"] >= 4, "Adequate (>=4 visits)", "Inadequate (<4 visits)"
)

# Map health facility delivery (FACDEL: 0=Home, 1=Health Facility)
df_clean["delivery_place"] = df_clean["FACDEL"].map(
    {0: "Home", 1: "Health Facility"}
)

# Map WASH indicators (WSIMPROVED and SANIMPROVED: 0=Unimproved, 1=Improved)
df_clean["water_source"] = df_clean["WSIMPROVED"].map(
    {0: "Unimproved", 1: "Improved"}
)
df_clean["sanitation"] = df_clean["SANIMPROVED"].map(
    {0: "Unimproved", 1: "Improved"}
)

# Map childhood diarrhea morbidity in past 2 weeks (CA1: 1=Yes, 2=No)
df_clean["recent_diarrhea"] = df_clean["CA1"].map({1: "Yes", 2: "No"})

print(f"Analytical cohort verified: N = {len(df_clean)}")
print(
    f"Overall stunting prevalence: {df_clean['STUNTED'].sum()} ({df_clean['STUNTED'].mean()*100:.2f}%)"
)


# ==============================================================================
# 2. GENERATE TABLE 1: BASELINE CHARACTERISTICS (UNWEIGHTED & WEIGHTED)
# ==============================================================================
def compute_table1_rows(data, var_name, category_order=None):
  rows = []
  total_n = len(data)
  total_w = data["chweight"].sum()

  val_counts_n = data[var_name].value_counts()
  val_counts_w = data.groupby(var_name)["chweight"].sum()

  cats = category_order if category_order else val_counts_n.index.tolist()
  for cat in cats:
    if cat in val_counts_n:
      n = val_counts_n[cat]
      pct_n = (n / total_n) * 100
      w_sum = val_counts_w[cat]
      pct_w = (w_sum / total_w) * 100
      rows.append({
          "Characteristic": f"  {cat}",
          "Unweighted n": f"{n:,}",
          "Unweighted %": f"{pct_n:.1f}%",
          "Survey-Weighted %": f"{pct_w:.1f}%",
      })
  return rows


table1_variables = [
    ("Child Nutritional Status (Outcome)", "STUNTED", [0, 1]),
    ("Child Sex", "sex", ["Male", "Female"]),
    ("Child Age Group", "age_group", age_labels),
    ("Area of Residence", "residence", ["Urban", "Rural"]),
    (
        "Geopolitical Zone",
        "zone",
        [
            "North Central",
            "North East",
            "North West",
            "South East",
            "South South",
            "South West",
        ],
    ),
    (
        "Maternal Education Level",
        "maternal_edu",
        ["None", "Primary", "Secondary", "Higher"],
    ),
    ("Maternal Age Category", "maternal_age_grp", mage_labels),
    (
        "Household Wealth Quintile",
        "wealth_quintile",
        ["Poorest", "Second", "Middle", "Fourth", "Richest"],
    ),
    (
        "Antenatal Care Utilization",
        "anc_cat",
        ["Adequate (>=4 visits)", "Inadequate (<4 visits)"],
    ),
    ("Place of Delivery", "delivery_place", ["Health Facility", "Home"]),
    ("Drinking Water Source", "water_source", ["Improved", "Unimproved"]),
    ("Sanitation Facility", "sanitation", ["Improved", "Unimproved"]),
    ("Recent Diarrhea (Past 2 Weeks)", "recent_diarrhea", ["No", "Yes"]),
]

table1_rows = []
for header, var, cats in table1_variables:
  table1_rows.append({
      "Characteristic": f"**{header}**",
      "Unweighted n": "",
      "Unweighted %": "",
      "Survey-Weighted %": "",
  })
  if var == "STUNTED":
    for c in cats:
      label = (
          "Stunted (HAZ < -2 SD)" if c == 1 else "Not Stunted (HAZ >= -2 SD)"
      )
      n = (df_clean[var] == c).sum()
      pct_n = (n / len(df_clean)) * 100
      w_sum = df_clean[df_clean[var] == c]["chweight"].sum()
      pct_w = (w_sum / df_clean["chweight"].sum()) * 100
      table1_rows.append({
          "Characteristic": f"  {label}",
          "Unweighted n": f"{n:,}",
          "Unweighted %": f"{pct_n:.1f}%",
          "Survey-Weighted %": f"{pct_w:.1f}%",
      })
  else:
    table1_rows.extend(compute_table1_rows(df_clean, var, cats))

table1_df = pd.DataFrame(table1_rows)
print("\n" + "=" * 50 + "\nTABLE 1: BASELINE CHARACTERISTICS\n" + "=" * 50)
print(table1_df.to_string(index=False))


# ==============================================================================
# 3. GENERATE TABLE 2: BIVARIABLE CROSS-TABULATIONS WITH CHI-SQUARE TESTS
# ==============================================================================
table2_rows = []
bivar_features = [
    ("Child Sex", "sex", ["Male", "Female"]),
    ("Child Age Group", "age_group", age_labels),
    ("Area of Residence", "residence", ["Urban", "Rural"]),
    (
        "Geopolitical Zone",
        "zone",
        [
            "North Central",
            "North East",
            "North West",
            "South East",
            "South South",
            "South West",
        ],
    ),
    (
        "Maternal Education Level",
        "maternal_edu",
        ["None", "Primary", "Secondary", "Higher"],
    ),
    ("Maternal Age Category", "maternal_age_grp", mage_labels),
    (
        "Household Wealth Quintile",
        "wealth_quintile",
        ["Poorest", "Second", "Middle", "Fourth", "Richest"],
    ),
    (
        "Antenatal Care Visits",
        "anc_cat",
        ["Adequate (>=4 visits)", "Inadequate (<4 visits)"],
    ),
    ("Place of Delivery", "delivery_place", ["Health Facility", "Home"]),
    ("Drinking Water Source", "water_source", ["Improved", "Unimproved"]),
    ("Sanitation Facility", "sanitation", ["Improved", "Unimproved"]),
    ("Recent Diarrhea", "recent_diarrhea", ["No", "Yes"]),
]

for header, var, cats in bivar_features:
  ct = pd.crosstab(df_clean[var], df_clean["STUNTED"])
  chi2, p_val, dof, _ = stats.chi2_contingency(ct)
  p_str = "<0.001" if p_val < 0.001 else f"{p_val:.4f}"

  table2_rows.append({
      "Explanatory Variable": f"**{header}**",
      "Not Stunted n (%)": "",
      "Stunted n (%)": "",
      "Total n": "",
      "Chi2 (df)": f"{chi2:.2f} ({dof})",
      "p-value": p_str,
  })

  for c in cats:
    if c in ct.index:
      not_stunted = ct.loc[c, 0]
      stunted = ct.loc[c, 1]
      tot = not_stunted + stunted
      table2_rows.append({
          "Explanatory Variable": f"  {c}",
          "Not Stunted n (%)": (
              f"{not_stunted:,} ({(not_stunted/tot)*100:.1f}%)"
          ),
          "Stunted n (%)": f"{stunted:,} ({(stunted/tot)*100:.1f}%)",
          "Total n": f"{tot:,}",
          "Chi2 (df)": "",
          "p-value": "",
      })

table2_df = pd.DataFrame(table2_rows)
print("\n" + "=" * 50 + "\nTABLE 2: BIVARIABLE CONTINGENCY ANALYSIS\n" + "=" * 50)
print(table2_df.to_string(index=False))


# ==============================================================================
# 4. GENERATE TABLE 3: MULTIVARIABLE CLUSTER-ADJUSTED LOGISTIC REGRESSION
# ==============================================================================
# Model formula specifying biological and policy reference groups
formula = (
    "STUNTED ~ C(sex, Treatment('Female')) + "
    "C(age_group, Treatment('0–11 months')) + "
    "C(residence, Treatment('Urban')) + "
    "C(zone, Treatment('South West')) + "
    "C(maternal_edu, Treatment('Higher')) + "
    "C(wealth_quintile, Treatment('Richest')) + "
    "C(anc_cat, Treatment('Adequate (>=4 visits)')) + "
    "C(delivery_place, Treatment('Health Facility')) + "
    "C(water_source, Treatment('Improved')) + "
    "C(sanitation, Treatment('Improved')) + "
    "C(recent_diarrhea, Treatment('No'))"
)

# Fit logit model with primary sampling unit clustering (HH1 = 200 clusters)
logit_model = smf.logit(formula=formula, data=df_clean).fit(
    cov_type="cluster", cov_kwds={"groups": df_clean["HH1"]}
)

# Compile coefficients, Adjusted Odds Ratios, 95% CIs, and p-values
params = logit_model.params
conf = logit_model.conf_int()
pvalues = logit_model.pvalues
se = logit_model.bse

table3_df = pd.DataFrame({
    "Adjusted_Beta": params.round(3),
    "Robust_SE": se.round(3),
    "aOR": np.exp(params).round(2),
    "CI_Lower": np.exp(conf[0]).round(2),
    "CI_Upper": np.exp(conf[1]).round(2),
    "p_value": pvalues.apply(
        lambda p: "<0.001" if p < 0.001 else f"{p:.4f}"
    ),
})

print(
    "\n"
    + "=" * 50
    + "\nTABLE 3: MULTIVARIABLE LOGISTIC REGRESSION (CLUSTER ROBUST)\n"
    + "=" * 50
)
print(table3_df)


# ==============================================================================
# 5. GENERATE PUBLICATION-GRADE FOREST PLOT (FIGURE 1)
# ==============================================================================
# Extract covariates for visual plotting
plot_data = [
    ("Maternal Education: None vs Higher", 4.94, 2.82, 8.64),
    ("Maternal Education: Primary vs Higher", 2.87, 1.72, 4.77),
    ("Maternal Education: Secondary vs Higher", 1.96, 1.21, 3.18),
    ("Wealth: Poorest vs Richest", 7.08, 3.75, 13.37),
    ("Wealth: Second vs Richest", 4.10, 2.24, 7.52),
    ("Wealth: Middle vs Richest", 3.27, 1.82, 5.87),
    ("Wealth: Fourth vs Richest", 1.79, 1.03, 3.13),
    ("Child Age: 48–59 vs 0–11 mos", 1.68, 1.29, 2.21),
    ("Child Age: 36–47 vs 0–11 mos", 1.55, 1.19, 2.01),
    ("Child Age: 24–35 vs 0–11 mos", 1.56, 1.22, 2.00),
    ("Child Age: 12–23 vs 0–11 mos", 1.48, 1.13, 1.93),
    ("Diarrhea: Yes vs No", 1.17, 0.90, 1.52),
    ("Residence: Rural vs Urban", 1.06, 0.87, 1.30),
    ("Sanitation: Unimproved vs Improved", 1.01, 0.81, 1.27),
    ("Water: Unimproved vs Improved", 1.00, 0.82, 1.22),
    ("Child Sex: Male vs Female", 0.98, 0.83, 1.15),
    ("ANC Visits: <4 vs >=4", 0.98, 0.78, 1.24),
    ("Delivery: Home vs Facility", 0.89, 0.71, 1.12),
]

# Invert order for top-to-bottom descending display
labels = [x[0] for x in plot_data][::-1]
aORs = [x[1] for x in plot_data][::-1]
lowers = [x[2] for x in plot_data][::-1]
uppers = [x[3] for x in plot_data][::-1]
y_positions = np.arange(len(labels))

plt.figure(figsize=(10, 8), dpi=300)

# Reference vertical null line (OR = 1.0)
plt.axvline(x=1.0, color="gray", linestyle="--", linewidth=1.2, alpha=0.8)

# Compute asymmetric error bars
x_errors = [np.array(aORs) - np.array(lowers), np.array(uppers) - np.array(aORs)]

# Draw point estimates and 95% CI bars
plt.errorbar(
    aORs,
    y_positions,
    xerr=x_errors,
    fmt="s",
    color="#1A365D",
    ecolor="#2B6CB0",
    elinewidth=2,
    capsize=4,
    markersize=6,
    label="Adjusted Odds Ratio (95% CI)",
)

# Plot formatting
plt.xscale("log")
plt.xlim(0.4, 18.0)
plt.xticks(
    [0.5, 1.0, 2.0, 4.0, 8.0, 16.0],
    ["0.5", "1.0", "2.0", "4.0", "8.0", "16.0"],
    fontsize=10,
)
plt.yticks(y_positions, labels, fontsize=9.5)
plt.xlabel(
    "Adjusted Odds Ratio (log scale)", fontsize=11, fontweight="bold", labelpad=8
)
plt.title(
    "Figure 1: Multivariable Forest Plot of Determinants of Stunting (N ="
    " 2,870)",
    fontsize=12,
    fontweight="bold",
    pad=12,
)
plt.grid(axis="x", linestyle=":", alpha=0.6)

# Right-aligned data callouts
for i, (or_val, l, u) in enumerate(zip(aORs, lowers, uppers)):
  plt.text(
      18.5,
      i,
      f"{or_val:.2f} [{l:.2f}, {u:.2f}]",
      va="center",
      fontsize=8.5,
      color="#2D3748",
      family="monospace",
  )

plt.text(
    18.5,
    len(labels),
    "aOR [95% CI]",
    va="bottom",
    fontsize=9.5,
    fontweight="bold",
    color="#1A202C",
)
plt.subplots_adjust(left=0.38, right=0.82, top=0.92, bottom=0.08)

# Save figure to file
plt.savefig("forest_plot_stunting.png", bbox_inches="tight")
plt.close()

print(
    "\nPipeline executed successfully. High-resolution figure"
    " 'forest_plot_stunting.png' saved."
)

Analytical cohort verified: N = 2870
Overall stunting prevalence: 961 (33.48%)


C:\Users\USER\AppData\Local\Temp\ipykernel_14112\1741100007.py:112: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_counts_w = data.groupby(var_name)["chweight"].sum()
C:\Users\USER\AppData\Local\Temp\ipykernel_14112\1741100007.py:112: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  val_counts_w = data.groupby(var_name)["chweight"].sum()



TABLE 1: BASELINE CHARACTERISTICS
                        Characteristic Unweighted n Unweighted % Survey-Weighted %
**Child Nutritional Status (Outcome)**                                            
            Not Stunted (HAZ >= -2 SD)        1,909        66.5%             66.3%
                 Stunted (HAZ < -2 SD)          961        33.5%             33.7%
                         **Child Sex**                                            
                                  Male        1,382        48.2%             48.0%
                                Female        1,488        51.8%             52.0%
                   **Child Age Group**                                            
                           0–11 months          582        20.3%             19.8%
                          12–23 months          561        19.5%             19.4%
                          24–35 months          571        19.9%             20.2%
                          36–47 months          596 

In [6]:
import matplotlib.pyplot as plt


def export_df_to_png(
    df, filename, note="", figsize=(10, 14)
):
  fig, ax = plt.subplots(figsize=figsize, dpi=300)
  ax.axis("off")

  table = ax.table(
      cellText=df.values,
      colLabels=df.columns,
      colWidths=[0.38, 0.12, 0.12, 0.10, 0.18, 0.10],
      loc="center",
      cellLoc="left",
  )
  table.auto_set_font_size(False)
  table.set_fontsize(8.5)
  table.scale(1, 1.35)

  # Header and border formatting
  for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("#D1D5DB")
    if row == 0:
      cell.set_facecolor("#1E3A8A")  # Navy header
      cell.set_text_props(weight="bold", color="#FFFFFF")
    else:
      if not str(df.iloc[row - 1, 0]).startswith("  "):
        cell.set_facecolor("#E2E8F0")  # Section subhead shading
        cell.set_text_props(weight="bold")
      else:
        cell.set_facecolor("#FFFFFF" if row % 2 == 0 else "#F8FAFC")
        if col > 0:
          cell.set_text_props(ha="right")

 
  plt.figtext(
      0.02, 0.01, note, fontsize=7.5, style="italic", wrap=True, color="#475569"
  )
  plt.savefig(filename, bbox_inches="tight")
  plt.close()


# call for Table 1:
export_df_to_png(
    table1_df,
    "table1.png",
    note="N = 2,870",
)

export_df_to_png(
    table2_df,
    "table2.png",
    note="N = 2,870",
)

export_df_to_png(
    table3_df,
    "table3.png",
    note="N = 2,870",
)

# Save figure to file
plt.savefig("table1_df.png", bbox_inches="tight")
plt.close()
print(
    "\nPipeline executed successfully. High-resolution figure"
    " 'table1_df.png' saved."
)

# Save figure to file
plt.savefig("table2_df.png", bbox_inches="tight")
plt.close()
print(
    "\nPipeline executed successfully. High-resolution figure"
    " 'table2_df.png' saved."
)

# Save figure to file
plt.savefig("table3_df.png", bbox_inches="tight")
plt.close()
print(
    "\nPipeline executed successfully. High-resolution figure"
    " 'table3_df.png' saved."
)


Pipeline executed successfully. High-resolution figure 'table1_df.png' saved.

Pipeline executed successfully. High-resolution figure 'table2_df.png' saved.

Pipeline executed successfully. High-resolution figure 'table3_df.png' saved.


In [12]:
import matplotlib.pyplot as plt
import pandas as pd

# Step 1: Create the DataFrame where 'Covariate' is an explicit column
table3_df = [
    ["Intercept", "-3.515", "0.315", "0.03", "[0.02, 0.06]", "<0.001"],
    ["Child Sex", "", "", "", "", ""],
    ["  Female", "Ref", "—", "1.00", "—", "—"],
    ["  Male", "-0.024", "0.085", "0.98", "[0.83, 1.15]", "0.781"],
    ["Child Age Group", "", "", "", "", ""],
    ["  0–11 months", "Ref", "—", "1.00", "—", "—"],
    ["  12–23 months", "0.389", "0.138", "1.48", "[1.13, 1.93]", "0.005"],
    ["  24–35 months", "0.443", "0.126", "1.56", "[1.22, 2.00]", "<0.001"],
    ["  36–47 months", "0.437", "0.133", "1.55", "[1.19, 2.01]", "0.001"],
    ["  48–59 months", "0.521", "0.138", "1.68", "[1.29, 2.21]", "<0.001"],
    ["Area of Residence", "", "", "", "", ""],
    ["  Urban", "Ref", "—", "1.00", "—", "—"],
    ["  Rural", "0.060", "0.103", "1.06", "[0.87, 1.30]", "0.557"],
    ["Geopolitical Zone", "", "", "", "", ""],
    ["  South West", "Ref", "—", "1.00", "—", "—"],
    ["  North Central", "-0.106", "0.198", "0.90", "[0.61, 1.33]", "0.591"],
    ["  North East", "-0.096", "0.208", "0.91", "[0.60, 1.37]", "0.645"],
    ["  North West", "-0.005", "0.184", "1.00", "[0.69, 1.43]", "0.979"],
    ["  South East", "-0.182", "0.218", "0.83", "[0.54, 1.28]", "0.405"],
    ["  South South", "0.027", "0.214", "1.03", "[0.68, 1.56]", "0.899"],
    ["Maternal Education Level", "", "", "", "", ""],
    ["  Higher", "Ref", "—", "1.00", "—", "—"],
    ["  Secondary", "0.672", "0.247", "1.96", "[1.21, 3.18]", "0.006"],
    ["  Primary", "1.053", "0.261", "2.87", "[1.72, 4.77]", "<0.001"],
    ["  None", "1.597", "0.286", "4.94", "[2.82, 8.64]", "<0.001"],
    ["Household Wealth Quintile", "", "", "", "", ""],
    ["  Richest", "Ref", "—", "1.00", "—", "—"],
    ["  Fourth", "0.584", "0.283", "1.79", "[1.03, 3.13]", "0.039"],
    ["  Middle", "1.183", "0.299", "3.27", "[1.82, 5.87]", "<0.001"],
    ["  Second", "1.411", "0.309", "4.10", "[2.24, 7.52]", "<0.001"],
    ["  Poorest", "1.958", "0.324", "7.08", "[3.75, 13.37]", "<0.001"],
    ["Antenatal Care Utilization", "", "", "", "", ""],
    ["  Adequate (>=4 visits)", "Ref", "—", "1.00", "—", "—"],
    ["  Inadequate (<4 visits)", "-0.018", "0.117", "0.98", "[0.78, 1.24]", "0.880"],
    ["Place of Delivery", "", "", "", "", ""],
    ["  Health Facility", "Ref", "—", "1.00", "—", "—"],
    ["  Home", "-0.111", "0.115", "0.89", "[0.71, 1.12]", "0.335"],
    ["Drinking Water Source", "", "", "", "", ""],
    ["  Improved", "Ref", "—", "1.00", "—", "—"],
    ["  Unimproved", "-0.001", "0.101", "1.00", "[0.82, 1.22]", "0.995"],
    ["Sanitation Facility", "", "", "", "", ""],
    ["  Improved", "Ref", "—", "1.00", "—", "—"],
    ["  Unimproved", "0.010", "0.116", "1.01", "[0.81, 1.27]", "0.928"],
    ["Recent Diarrhea (Past 2 Weeks)", "", "", "", "", ""],
    ["  No", "Ref", "—", "1.00", "—", "—"],
    ["  Yes", "0.154", "0.135", "1.17", "[0.90, 1.52]", "0.253"],
]

headers = ["Covariate", "Adjusted β", "Robust SE", "aOR", "95% CI", "p-value"]
df = pd.DataFrame(table3_df, columns=headers)

# Step 2: Render with explicit margin coordinates
fig, ax = plt.subplots(figsize=(12, 17), dpi=300)
ax.axis("off")

# We give colWidths dedicated space: 0.35 to Covariate
table = ax.table(
    cellText=df.values,
    colLabels=df.columns,
    colWidths=[0.35, 0.12, 0.12, 0.10, 0.18, 0.13],
    loc="center",
    cellLoc="left",
    bbox=[
        0.08,
        0.04,
        0.88,
        0.92,
    ],  # 0.08 adds a generous 8% left margin buffer
)

table.auto_set_font_size(False)
table.set_fontsize(8.5)

# Styling: colors and alignments
for (row, col), cell in table.get_celld().items():
  cell.set_edgecolor("#CBD5E1")
  if row == 0:
    cell.set_facecolor("#1E3A8A")  # Navy header
    cell.set_text_props(weight="bold", color="#FFFFFF")
  else:
    first_cell_text = str(df.iloc[row - 1, 0])
    # Main group headers (unindented) get light gray fill
    if not first_cell_text.startswith("  "):
      cell.set_facecolor("#E2E8F0")
      cell.set_text_props(weight="bold")
    else:
      cell.set_facecolor("#FFFFFF" if row % 2 == 0 else "#F8FAFC")
      if col > 0:
        cell.set_text_props(ha="right")

plt.figtext(
    0.08,
    0.01,
    "Note: N = 2,870; Clusters = 200. aOR = Adjusted Odds Ratio; CI = Confidence"
    " Interval.",
    fontsize=8,
    style="italic",
    color="#475569",
)

# Step 3: Save to file
plt.savefig("table3_final_fixed.png", dpi=300)
plt.close()
print("Saved successfully as 'table3_final_fixed.png'!")

Saved successfully as 'table3_final_fixed.png'!
